In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
OUTPUT_DIR = ROOT / "results" / "model_evaluation"

sys.path.insert(0, str(ROOT))
from src.utils.features import split_features

reference_df = pd.read_parquet(DATA_DIR / "cicids2017_train.parquet")
numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features

TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    # "unsw_nb15": ["unsw_nb15"],
    # "iot23": ["iot23"],
    # "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    # "cicids2017_iot23": ["cicids2017", "iot23"],
    # "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

TEST_DATASETS = [
    "cicids2017",
    "iot23",
    "unsw_nb15",
]

MODEL_CLASSES = {
    "decision_tree": DecisionTreeClassifier,
    "random_forest": RandomForestClassifier,
    "xgboost": XGBClassifier,
    "linear_svm": LinearSVC,
    "mlp": MLPClassifier,
}

MODELS_TO_RUN = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

In [2]:
columns = feature_columns + ["label_binary"]

test_sets = {
    name: pd.read_parquet(DATA_DIR / f"{name}_test.parquet", columns=columns)
    for name in TEST_DATASETS
}

for training_name, dataset_names in TRAINING_CONFIGS.items():
    train = pd.concat(
        [
            pd.read_parquet(
                DATA_DIR / f"{name}_train.parquet",
                columns=columns,
            )
            for name in dataset_names
        ],
        ignore_index=True,
    )

    for model_name in MODELS_TO_RUN:
        saved_pipeline = joblib.load(
            ROOT / "models" / model_name / f"{training_name}.joblib"
        )
        baseline_pipeline = clone(saved_pipeline)
        baseline_pipeline.set_params(
            classifier=MODEL_CLASSES[model_name](random_state=1)
        )

        report_dir = OUTPUT_DIR / training_name / model_name
        report_dir.mkdir(parents=True, exist_ok=True)

        for version, version_label, pipeline in [
            ("baseline", "padrão", baseline_pipeline),
            ("saved", "selecionado pela busca", saved_pipeline),
        ]:
            if version == "baseline":
                pipeline.fit(
                    train[feature_columns],
                    train["label_binary"].eq("ATTACK").astype("int8"),
                )

            for test_name, test in test_sets.items():
                report = classification_report(
                    test["label_binary"].eq("ATTACK").astype("int8"),
                    pipeline.predict(test[feature_columns]),
                    labels=[0, 1],
                    target_names=["BENIGN", "ATTACK"],
                    digits=4,
                    zero_division=0,
                )
                title = (
                    f"Treino: {training_name} | Modelo: {model_name} | "
                    f"Configuração: {version_label} | Teste: {test_name}"
                )
                report_path = report_dir / f"{version}_{test_name}.txt"
                report_path.write_text(f"{title}\n\n{report}", encoding="utf-8")


In [3]:
def show_reports(training_name, model_name):
    report_dir = OUTPUT_DIR / training_name / model_name
    for test_name in TEST_DATASETS:
        for version in ("baseline", "saved"):
            report_path = report_dir / f"{version}_{test_name}.txt"
            print(report_path.read_text(encoding="utf-8"))


In [4]:
show_reports("cicids2017", "decision_tree")


Treino: cicids2017 | Modelo: decision_tree | Configuração: padrão | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9972    0.9953    0.9963     80740
      ATTACK     0.9815    0.9889    0.9852     20185

    accuracy                         0.9941    100925
   macro avg     0.9894    0.9921    0.9907    100925
weighted avg     0.9941    0.9941    0.9941    100925

Treino: cicids2017 | Modelo: decision_tree | Configuração: selecionado pela busca | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9972    0.9953    0.9963     80740
      ATTACK     0.9815    0.9889    0.9852     20185

    accuracy                         0.9941    100925
   macro avg     0.9894    0.9921    0.9907    100925
weighted avg     0.9941    0.9941    0.9941    100925

Treino: cicids2017 | Modelo: decision_tree | Configuração: padrão | Teste: iot23

              precision    recall  f1-score   support

      BENIGN     0.

In [5]:
show_reports("cicids2017", "random_forest")


Treino: cicids2017 | Modelo: random_forest | Configuração: padrão | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9983    0.9952    0.9968     80740
      ATTACK     0.9811    0.9934    0.9872     20185

    accuracy                         0.9948    100925
   macro avg     0.9897    0.9943    0.9920    100925
weighted avg     0.9949    0.9948    0.9949    100925

Treino: cicids2017 | Modelo: random_forest | Configuração: selecionado pela busca | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9984    0.9952    0.9968     80740
      ATTACK     0.9811    0.9935    0.9872     20185

    accuracy                         0.9949    100925
   macro avg     0.9897    0.9943    0.9920    100925
weighted avg     0.9949    0.9949    0.9949    100925

Treino: cicids2017 | Modelo: random_forest | Configuração: padrão | Teste: iot23

              precision    recall  f1-score   support

      BENIGN     0.

In [6]:
show_reports("cicids2017", "xgboost")


Treino: cicids2017 | Modelo: xgboost | Configuração: padrão | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9985    0.9947    0.9966     80740
      ATTACK     0.9793    0.9940    0.9866     20185

    accuracy                         0.9946    100925
   macro avg     0.9889    0.9943    0.9916    100925
weighted avg     0.9946    0.9946    0.9946    100925

Treino: cicids2017 | Modelo: xgboost | Configuração: selecionado pela busca | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9985    0.9947    0.9966     80740
      ATTACK     0.9793    0.9940    0.9866     20185

    accuracy                         0.9946    100925
   macro avg     0.9889    0.9943    0.9916    100925
weighted avg     0.9946    0.9946    0.9946    100925

Treino: cicids2017 | Modelo: xgboost | Configuração: padrão | Teste: iot23

              precision    recall  f1-score   support

      BENIGN     0.3193    0.1161    

In [7]:
show_reports("cicids2017", "linear_svm")


Treino: cicids2017 | Modelo: linear_svm | Configuração: padrão | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.8972    0.9714    0.9329     80740
      ATTACK     0.8292    0.5550    0.6649     20185

    accuracy                         0.8881    100925
   macro avg     0.8632    0.7632    0.7989    100925
weighted avg     0.8836    0.8881    0.8793    100925

Treino: cicids2017 | Modelo: linear_svm | Configuração: selecionado pela busca | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9360    0.9746    0.9549     80740
      ATTACK     0.8784    0.7333    0.7993     20185

    accuracy                         0.9264    100925
   macro avg     0.9072    0.8540    0.8771    100925
weighted avg     0.9244    0.9264    0.9238    100925

Treino: cicids2017 | Modelo: linear_svm | Configuração: padrão | Teste: iot23

              precision    recall  f1-score   support

      BENIGN     0.3351    0

In [8]:
show_reports("cicids2017", "mlp")


Treino: cicids2017 | Modelo: mlp | Configuração: padrão | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9979    0.9942    0.9960     80740
      ATTACK     0.9771    0.9916    0.9843     20185

    accuracy                         0.9937    100925
   macro avg     0.9875    0.9929    0.9902    100925
weighted avg     0.9937    0.9937    0.9937    100925

Treino: cicids2017 | Modelo: mlp | Configuração: selecionado pela busca | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9978    0.9943    0.9961     80740
      ATTACK     0.9775    0.9913    0.9844     20185

    accuracy                         0.9937    100925
   macro avg     0.9877    0.9928    0.9902    100925
weighted avg     0.9938    0.9937    0.9937    100925

Treino: cicids2017 | Modelo: mlp | Configuração: padrão | Teste: iot23

              precision    recall  f1-score   support

      BENIGN     0.7014    0.1113    0.1921     8